# DeepTuneIO — Module 04: Access Pattern - DXT Trace Processor

Uses common `DATASET_FILTER`: `exact`, `prefix`, `extension`, `regex`.


### Module 04 — OST wrap-around

Se distingue la observación compacta DXT de la secuencia completa de stripes. Esto corrige peticiones que vuelven a un OST dentro de la misma operación.

- `OSTs_Observed_DXT`: observación DXT.
- `OSTs_Predicted_Layout`: OST únicos predichos para validación.
- `OST_Stripe_Sequence`: recorrido completo con repeticiones.
- `OST_Logical_Distribution`: bytes lógicos exactos por OST.
- `OST_Byte_Conservation`: conservación de bytes.


### Module 04 — dataset filesystem fix

The processor now distinguishes the filesystem of the current DXT block from the filesystem of the selected dataset. This prevents auxiliary DXT records appearing later in a trace (for example files on `ext4`) from overwriting the dataset filesystem (`lustre`) and disabling OST mapping.


## Logical OST workload mapping 

For Lustre traces, Module 04 now derives the logical byte distribution of each DXT operation over the OST IDs reported by Darshan. The mapping uses `Offset(bytes)`, `Request_Size(bytes)` and `Lustre stripe_size` rather than the former `Request_Size // stripe_size` approximation.

The result is explicitly a **logical file-layout workload estimate**, not a measurement of physical storage-device traffic after caching, readahead or RPC aggregation.

Additional products per DXT trace:
- `*_dxt_analysis.csv`: compact per-operation mapping columns;
- `*_ost_workload.csv`: Sankey-ready aggregation by Jobid, node, rank, operation and OST.

The mapping validates byte conservation for every operation. NFS does not perform OST mapping.


### Compatibilidad de metadatos Lustre 

Se reconocen y normalizan ambos formatos DXT históricos:

```text
# DXT, Lustre stripe_size: 1048576, Lustre stripe_count: 1
# DXT, Lustre OST obdidx: 14
```

y:

```text
# DXT, Lustre stripe components:
# [Component 1] stripe_ext: 0 - EOF, stripe_size: 1048576, stripe_count: 1, OSTs: 14
```

Ambos se convierten a componentes Lustre normalizados. También queda preparado el soporte para múltiples componentes/PFL según el offset de cada operación.


### Organización de salidas 

Los productos de Module 06 se separan según su función analítica:

```text
results/Access_Pattern_DXT/
├── Traces/        # *_dxt.csv: traza DXT estructurada
├── Analysis/      # *_dxt_analysis.csv: métricas derivadas por operación
├── OST_Workload/  # *_ost_workload.csv: carga lógica por OST / Sankey
└── Summary/       # resúmenes por proceso y ejecución
```

Los ficheros existentes se sobrescriben al reprocesar la misma traza; no se agregan filas a resultados anteriores.


### OST mapping validado contra DXT 

La lista `[OST]` de cada operación DXT es la fuente primaria para identificar los OST realmente reportados por Darshan. `Offset + Request_Size + stripe_size` se usan para repartir los bytes entre esos OST observados.

El layout Lustre completo se utiliza separadamente para predecir la secuencia esperada y validar el DXT.

Nuevas columnas:
- `OSTs_Observed_DXT`
- `OSTs_Predicted_Layout`
- `OST_Layout_Match`
- `OST_Logical_Distribution`
- `OST_Logical_Bytes_Total`

Además, el layout se captura únicamente para el dataset definido por `DATASET_FILTER`; los layouts de módulos Python y otros ficheros auxiliares ya no contaminan el análisis.


### Layout Lustre por fichero \n\nEl layout Lustre se almacena por cada fichero del dataset. Esto generaliza Module 04 para `shared` y `file_per_process`.\n\nCada operación usa `File_name` para seleccionar su layout. `OSTs_Observed_DXT` sigue siendo la fuente primaria y `OSTs_Predicted_Layout` se calcula con el layout específico del fichero. `LUSTRE_Layout_File` registra qué layout fue utilizado.\n

In [1]:
from pathlib import Path
import sys
from importlib.util import spec_from_file_location, module_from_spec

MODULE_DIR = Path("./Module").resolve()
MODULE_FILE = MODULE_DIR / "04_process_dxt_traces.py"

spec = spec_from_file_location("deeptuneio_module04", MODULE_FILE)
module04 = module_from_spec(spec)
sys.modules[spec.name] = module04
spec.loader.exec_module(module04)

print("Module:", MODULE_FILE)
print("Version:", module04.VERSION)


Module: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\06_DXT_Processing\DeepTuneIO_Module06_DXT_FILTER_v5_6\Module\06_process_dxt_traces.py
Version: 5.6.0


## 1. Selección de campaña y filtro


In [2]:
CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": ["lustre_1ost", "lustre_2ost", "lustre_4ost", "nfs_SnGPU_e1"],
            "file_format": ".hdf5",
            "dataset_filter": {"mode": "exact", "value": "output_bw_512.hdf5"},
        },
    },
    "DLIOv1": {
        "HDF5_64bs": {"scenarios": ["lustre_ost"], "file_format": ".h5",
                      "dataset_filter": {"mode": "prefix", "value": "img_"}},
        "NPZ_64bs": {"scenarios": ["lustre_ost"], "file_format": ".npz",
                    "dataset_filter": {"mode": "prefix", "value": "img_"}},
        "TFRecord_256kts_64bs": {"scenarios": ["lustre_ost"], "file_format": ".tfrecords",
                                "dataset_filter": {"mode": "prefix", "value": "img_"}},
        "TFRecord_1mts_64bs": {"scenarios": ["lustre_ost"], "file_format": ".tfrecords",
                              "dataset_filter": {"mode": "prefix", "value": "img_"}},
    },
}

APPLICATION_POS = 1
DATASET_POS = 1
SCENARIO_POS = 4

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
SCENARIO = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"][SCENARIO_POS - 1]
FILE_FORMAT = CAMPAIGNS[APPLICATION][DATASET_GROUP]["file_format"]
DATASET_FILTER = CAMPAIGNS[APPLICATION][DATASET_GROUP]["dataset_filter"]

print("Application   :", APPLICATION)
print("Dataset group :", DATASET_GROUP)
print("Scenario      :", SCENARIO)
print("File format   :", FILE_FORMAT)
print("Filter mode   :", DATASET_FILTER["mode"])
print("Filter value  :", DATASET_FILTER["value"])


Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : nfs_SnGPU_e1
File format   : .hdf5
Filter mode   : exact
Filter value  : output_bw_512.hdf5


In [3]:
PROJECT_ROOT = Path(r"F:\NoteBooks\CORA_DR_UAB").resolve()
ARTICLE_ID = "Article_01_Parallel_IO_Analysis"
CAMPAIGN_ROOT = PROJECT_ROOT / ARTICLE_ID / "Data" / APPLICATION / DATASET_GROUP / SCENARIO

print("Campaign root:", CAMPAIGN_ROOT)
print("Exists       :", CAMPAIGN_ROOT.exists())


Campaign root: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
Exists       : True


## 2. Ejecutar Module 04


In [4]:
return_code = module04.main([
    "--campaign-root", str(CAMPAIGN_ROOT),
    "--application", APPLICATION,
    "--dataset-group", DATASET_GROUP,
    "--scenario", SCENARIO,
    "--file-format", FILE_FORMAT,
    "--filter-mode", DATASET_FILTER["mode"],
    "--filter-value", DATASET_FILTER["value"],
    "--event", "JSC24",
    "--module-root", str(MODULE_DIR),
])
print("Return code:", return_code)


DeepTuneIO — Module 06: DXT Trace Processor
Campaign root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1
Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : nfs_SnGPU_e1
File format   : .hdf5
Filter mode   : exact
Filter value  : output_bw_512.hdf5
DXT input     : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\processed\DXT
DXT output    : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Traces
Summary dir   : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Summary


Análisis DXT:   0%|          | 0/25 [00:00<?, ?archivo/s, Fichero actual: 10_DG_N8p32e1_EfficientNetB4_outputbw512...]

Archivo de análisis detallado guardado: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Analysis\10_DG_N8p32e1_EfficientNetB4_outputbw512hdf5_ds36nc14_M0_nfs_5181620_FT3RESDT_dxt_analysis.csv
Resumen global guardado con éxito en: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Summary\3_Summary_Global_DeepGalaxy_DXT_hdf5_nfs_SnGPU_e1_JSC24.csv
Resumen por proceso guardado con éxito en: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Summary\3_Summary_Process_DeepGalaxy_DXT_hdf5_nfs_SnGPU_e1_JSC24.csv


Análisis DXT:   4%|▍         | 1/25 [00:09<03:39,  9.14s/archivo, Fichero actual: 11_DG_N12p48e1_EfficientNetB4_outputbw51...]

Archivo de análisis detallado guardado: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Analysis\11_DG_N12p48e1_EfficientNetB4_outputbw512hdf5_ds36nc14_M0_nfs_5006144_FT3RESDT_dxt_analysis.csv
Resumen global guardado con éxito en: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Summary\3_Summary_Global_DeepGalaxy_DXT_hdf5_nfs_SnGPU_e1_JSC24.csv


...

Resumen por proceso guardado con éxito en: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Summary\3_Summary_Process_DeepGalaxy_DXT_hdf5_nfs_SnGPU_e1_JSC24.csv
Archivo de análisis detallado guardado: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Analysis\8_DG_N8p32e1_EfficientNetB4_outputbw512hdf5_ds36nc14_M0_nfs_5173308_FT3RESDT_dxt_analysis.csv
Resumen global guardado con éxito en: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Summary\3_Summary_Global_DeepGalaxy_DXT_hdf5_nfs_SnGPU_e1_JSC24.csv


Análisis DXT:  96%|█████████▌| 24/25 [05:37<00:11, 11.36s/archivo, Fichero actual: 9_DG_N8p32e1_EfficientNetB4_outputbw512h...]

Resumen por proceso guardado con éxito en: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Summary\3_Summary_Process_DeepGalaxy_DXT_hdf5_nfs_SnGPU_e1_JSC24.csv
Archivo de análisis detallado guardado: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Analysis\9_DG_N8p32e1_EfficientNetB4_outputbw512hdf5_ds36nc14_M0_nfs_5179061_FT3RESDT_dxt_analysis.csv
Resumen global guardado con éxito en: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Summary\3_Summary_Global_DeepGalaxy_DXT_hdf5_nfs_SnGPU_e1_JSC24.csv


Análisis DXT: 100%|██████████| 25/25 [05:47<00:00, 13.89s/archivo, Fichero actual: 9_DG_N8p32e1_EfficientNetB4_outputbw512h...]

Resumen por proceso guardado con éxito en: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\nfs_SnGPU_e1\results\Access_Pattern_DXT\Summary\3_Summary_Process_DeepGalaxy_DXT_hdf5_nfs_SnGPU_e1_JSC24.csv
✔ ¡Procesamiento completado! Se analizaron con éxito 25 archivos.
📊 Reportes consolidados y guardados en la raíz de resultados.
Return code: 0
